In [1]:
#Based on Modern Robotics Library 
import numpy as np 
import modern_robotics as mr


In [2]:
## NextState uses kinematics of robot
## current_state must have the following input format in a list: np.array([x1,x2,x3...x12]) consisting of [chassis:1,2,3, arm:1,2,3,4,5 , wheel angle:1,2,3,4]
## velocities must be np.array([v1,v2,...]) consisting of [arm:1,2,3,4,5 , wheel: 1,2,3,4]
## time_step t
## magnitude of maximum joint and wheel velocity magnitude 
## returns output of the new (next) chassis state with chassis, arm, wheel states
def NextState(current_state, velocities,time_step,magnitude):

    #clipping veloctities to keep with joint and wheel limits 
    velocities = np.clip(velocities, -magnitude, magnitude)

    #extracting states
    chassis_current_state = current_state[0:3]
    arm_1_current_state, arm_2_current_state, arm_3_current_state, arm_4_current_state, arm_5_current_state = current_state[3:8]
    wheel_1_current_state, wheel_2_current_state, wheel_3_current_state, wheel_4_current_state = current_state[8:12]
    
    #extracting velocities
    arm_1_current_velocity, arm_2_current_velocity, arm_3_current_velocity, arm_4_current_velocity, arm_5_current_velocity = velocities[0:5]
    wheel_1_current_velocity, wheel_2_current_velocity, wheel_3_current_velocity, wheel_4_current_velocity = velocities[5:]
    
    
    #NextState implementation
    #arm
    arm_total_state_vec = np.array([arm_1_current_state, arm_2_current_state, arm_3_current_state, arm_4_current_state, arm_5_current_state])
    arm_total_vel_vec = np.array([arm_1_current_velocity, arm_2_current_velocity, arm_3_current_velocity, arm_4_current_velocity, arm_5_current_velocity])

    new_arm_state = arm_total_state_vec + arm_total_vel_vec*time_step 


    #wheel
    wheel_total_state_vec = np.array([wheel_1_current_state, wheel_2_current_state, wheel_3_current_state, wheel_4_current_state])
    wheel_total_vel_vec = np.array([wheel_1_current_velocity, wheel_2_current_velocity, wheel_3_current_velocity, wheel_4_current_velocity])

    new_wheel_state = wheel_total_state_vec + wheel_total_vel_vec*time_step 

    #chassis 
    #defining kinematics 
    l = 0.47/2 #m
    w = 0.3/2 #m
    r = 0.0475 #m
    
    ##odometry 
    # Vb = F * (wheel_vel*t)
    Vb = r/4 * np.array([[-1/(l+w), 1/(l+w), 1/(l+w), -1/(l+w)], [1,1,1,1], [-1,1,-1,1]]) @ np.array([wheel_total_vel_vec[0]*time_step, wheel_total_vel_vec[1]*time_step, wheel_total_vel_vec[2]*time_step, wheel_total_vel_vec[3]*time_step])
    wbz, vbx, vby = Vb 
    
    #dq calculations
    if wbz < 10**-5:
        dqb = np.array([0, vbx, vby])
    else:
        dqb = np.array([wbz, (vbx*np.sin(wbz) + vby*(np.cos(wbz) -1))/wbz, (vby*np.sin(wbz) + vbx*(-np.cos(wbz)+1))/wbz])

    dq = np.array([[1,0,0], [0, np.cos(chassis_current_state[0]), -np.sin(chassis_current_state[0])], [0, np.sin(chassis_current_state[0]), np.cos(chassis_current_state[0])]]) @ dqb

    new_chassis_odom_state = np.array([chassis_current_state[0], chassis_current_state[1], chassis_current_state[2]]) + dq 
    
    return np.concatenate([new_chassis_odom_state, new_arm_state, new_wheel_state])

In [3]:
#milestone 2 running
velocities=np.array([0.1,0.1,0.1,0.1,1,1,1,1,1])
magnitude=100
dt=0.01
full_predictions=[]
kinematic_prediction=np.array([0,0,0,0,0,0,0,0,0,0,0,0,0])
i=0
while i < 500:
    kinematic_prediction=NextState(kinematic_prediction,velocities,dt,magnitude)
    kinematic_prediction=np.hstack([kinematic_prediction,1])
    i=i+1
    full_predictions.append(kinematic_prediction)

#for i in result:
#    kinematic_prediction=NextState(i,velocities,dt,magnitude)
#    kinematic_prediction=np.hstack([kinematic_prediction,1])

#    full_predictions.append(kinematic_prediction)
np.savetxt("milestone_2_test.csv",full_predictions,delimiter=",")